# XGBoost Champion Model & Final Tasks


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    BASE = "/content/drive/MyDrive/ColabNotebooks/ALDA"
    DATA_PATH = os.path.join(BASE, 'data', 'train.csv')
    RESULTS_DIR = os.path.join(BASE, 'econet', 'results')
    print("Running in Google Colab.")
except ImportError:
    print("Not running in Colab. Using local data path.")
    import os
    DATA_PATH = "data/train.csv"
    RESULTS_DIR = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, average_precision_score, fbeta_score, confusion_matrix, precision_recall_curve, auc

# --- DATA LOADER ---
def load_econet_data(file_path):
    try:
        print(f"Loading data from {file_path}...")
        df = pd.read_csv(file_path, parse_dates=['Ob'])
        if 'Ob' in df.columns:
            df = df.sort_values(by=['Ob', 'Station']).reset_index(drop=True)
        return df
    except FileNotFoundError:
        print(f"File not found at {file_path}. Please check drive mounting.")
        return pd.DataFrame()

# --- FEATURE ENGINEERING ---
def compute_novel_features(df, time_col='Ob', station_col='Station', temp_col='value'):
    df_feat = df.copy()
    df_feat = df_feat.sort_values(by=[station_col, time_col]).reset_index(drop=True)

    # 1. temp_roc
    df_feat['temp_roc'] = df_feat.groupby(station_col)[temp_col].diff()
    df_feat['temp_roc'] = df_feat['temp_roc'].fillna(0)

    # 2. buddy_dev
    mean_temp_timestamp = df_feat.groupby(time_col)[temp_col].transform('mean')
    df_feat['buddy_dev'] = df_feat[temp_col] - mean_temp_timestamp
    df_feat['buddy_dev'] = df_feat['buddy_dev'].fillna(0)
    
    # 3. Time components
    if df_feat[time_col].dtype == 'object':
        df_feat[time_col] = pd.to_datetime(df_feat[time_col])
    df_feat['hour'] = df_feat[time_col].dt.hour
    df_feat['day_of_week'] = df_feat[time_col].dt.dayofweek
    df_feat['month'] = df_feat[time_col].dt.month

    # 4. QC Flag aggregates
    flag_cols = ['R_flag', 'I_flag', 'Z_flag', 'B_flag']
    existing_flags = [c for c in flag_cols if c in df_feat.columns]
    if existing_flags:
        active_flags = (df_feat[existing_flags] != 0).astype(int)
        df_feat['flag_sum'] = active_flags.sum(axis=1)
        df_feat['any_flag'] = (df_feat['flag_sum'] > 0).astype(int)
    else:
        df_feat['flag_sum'] = 0
        df_feat['any_flag'] = 0

    return df_feat

novel_feature_transformer = FunctionTransformer(compute_novel_features, kw_args={})
log1p_transformer = FunctionTransformer(np.log1p, validate=False)

# --- PIPELINE BUILDER ---
def get_preprocessing_pipeline(numeric_features=None, skewed_features=None, categorical_features=None, use_scaler=True):
    if numeric_features is None: 
        numeric_features = ['value', 'temp_roc', 'buddy_dev', 'R_flag', 'I_flag', 'Z_flag', 'B_flag', 'hour', 'day_of_week', 'month', 'flag_sum', 'any_flag']
    if skewed_features is None: skewed_features = []
    if categorical_features is None: categorical_features = ['Station', 'measure']

    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if use_scaler:
        num_steps.append(('scaler', StandardScaler()))
    numeric_transformer = Pipeline(steps=num_steps)

    skew_steps = [('imputer', SimpleImputer(strategy='median')), ('log1p', log1p_transformer)]
    if use_scaler:
        skew_steps.append(('scaler', StandardScaler()))
    skewed_transformer = Pipeline(steps=skew_steps)

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('skewed', skewed_transformer, skewed_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop'
    )
    return preprocessor

# --- SPLITTING ---
def temporal_train_test_split(df, target_col='target', time_col='Ob', test_size=0.2):
    if df.empty:
        raise ValueError("DataFrame is empty.")
    df_sorted = df.sort_values(by=time_col).reset_index(drop=True)
    split_idx = int(len(df_sorted) * (1 - test_size))
    train_df = df_sorted.iloc[:split_idx]
    test_df = df_sorted.iloc[split_idx:]
    
    return train_df.drop(columns=[target_col]), test_df.drop(columns=[target_col]), train_df[target_col], test_df[target_col]

# --- EVALUATION METRICS SAVER ---
def save_evaluation_metrics(model_name, y_true, y_pred, y_proba):
    print(f"\n=== {model_name} Performance ===")
    out_dir = os.path.join(RESULTS_DIR, model_name.replace(' ', '_').lower())
    os.makedirs(out_dir, exist_ok=True)
    
    f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    pr_auc = average_precision_score(y_true, y_proba)
    
    metrics = {
        'F2': f2,
        'PR-AUC': pr_auc,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    }
    
    with open(os.path.join(out_dir, f'{model_name}_metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=4)
        
    print(f"F2 Score: {f2:.4f}")
    print(f"PR-AUC:   {pr_auc:.4f}")
    print(f"Results saved to {out_dir}")
    return metrics


In [ ]:
# Load Data
df_raw = load_econet_data(DATA_PATH)
if not df_raw.empty:
    print(f"Raw shape: {df_raw.shape}")

    print("Engineering features (including buddy_dev & temp_roc)...")
    df_eng = compute_novel_features(df_raw)

    print("Splitting chronologically...")
    X_train, X_test, y_train, y_test = temporal_train_test_split(df_eng, target_col='target', time_col='Ob')
    print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
else:
    print("DataFrame empty, skipping splits.")



In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
import shap

if not df_raw.empty:
    categorical_features = ['Station', 'measure']
    enhanced_numeric = ['value', 'temp_roc', 'buddy_dev', 'R_flag', 'I_flag', 'Z_flag', 'B_flag', 'hour', 'day_of_week', 'month', 'flag_sum', 'any_flag']

    preprocessor = get_preprocessing_pipeline(enhanced_numeric, [], categorical_features, use_scaler=False)
    scale_w = ((y_train == 0).sum() / (y_train == 1).sum()) if (y_train == 1).sum() > 0 else 1

    xgb_base = xgb.XGBClassifier(scale_pos_weight=scale_w, eval_metric='aucpr', random_state=42, n_jobs=-1, tree_method='hist', device='cuda')
    pipeline = Pipeline([('preprocess', preprocessor), ('classifier', xgb_base)])
    tscv = TimeSeriesSplit(n_splits=5)

    param_search = {'classifier__learning_rate': [0.01, 0.05, 0.1, 0.3], 'classifier__max_depth': [3, 5, 7, 9], 'classifier__n_estimators': [100, 200, 500]}
    search = RandomizedSearchCV(pipeline, param_search, n_iter=5, scoring={'f2': make_scorer(fbeta_score, beta=2, zero_division=0), 'pr_auc': 'average_precision'}, refit='f2', cv=tscv, random_state=42)

    try:
        search.fit(X_train, y_train)
    except xgb.core.XGBoostError:
        print("CUDA missing, falling back to CPU")
        xgb_base.set_params(device='cpu')
        search.fit(X_train, y_train)

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    metrics = save_evaluation_metrics("XGBoost", y_test, y_pred, y_proba)
    out_dir = os.path.join(RESULTS_DIR, 'xgboost')
    joblib.dump(best_model, os.path.join(out_dir, 'xgb_model.pkl'))

    # Save winning hyperparameters and CV score
    out_dir = os.path.join(RESULTS_DIR, 'xgboost')
    os.makedirs(out_dir, exist_ok=True)
    tuning_results = {
        'best_parameters': search.best_params_,
        'best_cv_f2': float(search.best_score_)
    }
    with open(os.path.join(out_dir, 'hyperparameters.json'), 'w') as f:
        json.dump(tuning_results, f, indent=4)



## Task 1: Cost-Ratio Threshold Derivation


In [ ]:
if not df_raw.empty:
    from sklearn.metrics import classification_report
    
    thresholds = np.arange(0.1, 0.95, 0.05)
    f2_scores = []
    precisions = []
    recalls = []
    
    for t in thresholds:
        y_pred_t = (y_proba >= t).astype(int)
        f2_scores.append(fbeta_score(y_test, y_pred_t, beta=2, zero_division=0))
        
        from sklearn.metrics import precision_score, recall_score
        # Compute precision and recall directly to avoid string key mappings
        precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
        recalls.append(recall_score(y_test, y_pred_t, zero_division=0))

    opt_idx = np.argmax(f2_scores)
    optimal_threshold = thresholds[opt_idx]
    
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, f2_scores, label="F2 Score", color='green', marker='o', lw=2)
    plt.plot(thresholds, precisions, label="Precision", color='blue', linestyle=':', alpha=0.7)
    plt.plot(thresholds, recalls, label="Recall", color='red', linestyle='-.', alpha=0.7)

    plt.axvline(optimal_threshold, color='black', linestyle='-', lw=2, label=f'Optimal F2 Threshold ({optimal_threshold:.2f})')

    plt.title("Cost-Sensitive Threshold Sweep (F2, Precision, Recall)")
    plt.xlabel("Classification Decision Threshold")
    plt.ylabel("Metric Score")
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'cost_ratio_threshold_sweep.png'))
    plt.show()
    
    with open(os.path.join(out_dir, 'threshold_analysis.json'), 'w') as f:
        json.dump({'optimal_threshold': float(optimal_threshold), 'max_f2_score': float(np.max(f2_scores))}, f)

## Task 2 & 3: Final Confusion Matrices & Station SHAP Validation


In [ ]:
if not df_raw.empty:
    y_pred_optimal = (y_proba >= optimal_threshold).astype(int)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=False,
                xticklabels=['Valid', 'Error'], yticklabels=['Valid', 'Error'])
    ax1.set_title(f'Default Threshold (0.50)\nF2={metrics["F2"]:.4f}')
    ax1.set_xlabel('Predicted Label')
    ax1.set_ylabel('True Label')
    
    sns.heatmap(confusion_matrix(y_test, y_pred_optimal), annot=True, fmt='d', cmap='Blues', ax=ax2, cbar=False,
                xticklabels=['Valid', 'Error'], yticklabels=['Valid', 'Error'])
    ax2.set_title(f'Optimal Threshold ({optimal_threshold:.2f})\nF2={np.max(f2_scores):.4f}')
    ax2.set_xlabel('Predicted Label')
    ax2.set_ylabel('True Label')
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'final_confusion_matrices.png'))
    plt.show()

    xgb_estimator = best_model.named_steps['classifier']
    pre_pipe = best_model.named_steps['preprocess']
    
    X_test_transformed = pre_pipe.transform(X_test)
    feature_names = pre_pipe.get_feature_names_out()
    
    try:
        import shap
        shap.initjs()
        explainer = shap.TreeExplainer(xgb_estimator)
        shap_values = explainer.shap_values(X_test_transformed, check_additivity=False)
        
        shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, show=False)
        plt.title("Overall SHAP Validation", y=1.05)
        plt.savefig(os.path.join(out_dir, 'overall_shap_summary.png'), bbox_inches='tight')
        plt.show()
        
        stat_idx = list(feature_names).index('cat__Station') if 'cat__Station' in feature_names else -1
        if stat_idx != -1:
            shap.dependence_plot(stat_idx, shap_values, X_test_transformed, feature_names=feature_names, show=False)
            plt.title("Error Vulnerability by Station")
            plt.savefig(os.path.join(out_dir, 'shap_station_dependence.png'), bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"SHAP Exception: {e}")